# Detecção de Fraude em Transações — v2 (Google Colab)

Passo a passo interativo do pipeline. **Como usar no Colab:** faça upload da pasta do projeto (ou do `.zip`), descompacte e execute as células em ordem. Para dados reais, defina `CSV_PATH`.

In [ ]:
# 0. Dependências (uma vez). Opcionais (xgboost, lightgbm, catboost, shap) são usadas se instaladas.
# !pip install -q -r requirements.txt

In [ ]:
import os, sys, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
sys.path.insert(0, os.getcwd())
from src import data_gen, eda, splits, feature_engineering as fe, models, evaluation as ev, drift
CSV_PATH = None   # ex.: 'creditcard.csv'  (None = dados sintéticos)

In [ ]:
# 1. Dados + qualidade
df = data_gen.load_dataset(csv_path=CSV_PATH) if CSV_PATH else data_gen.generate_synthetic_transactions(n_samples=30_000)
print(splits.validate_dataset(df))
print(eda.check_data_quality(df)["linhas_duplicadas"], "duplicados |", eda.check_data_quality(df)["total_nulos"], "nulos")
eda.class_balance(df)

In [ ]:
# 2. Limpeza + features CAUSAIS (só passado) + split TEMPORAL
df = eda.clean_basic_issues(df)
dfe = fe.add_causal_features(df)          # sem 'id_cartao' cai para features globais
feats = [c for c in dfe.columns if c not in ("Class", "Time", "id_cartao")]
tr, va, te = splits.temporal_split(dfe)
X_tr, y_tr = tr[feats], tr["Class"].to_numpy()
X_va, y_va = va[feats], va["Class"].to_numpy()
X_te, y_te = te[feats], te["Class"].to_numpy()
med = tr["Amount"].median()
amt_va, amt_te = va["Amount"].fillna(med).to_numpy(), te["Amount"].fillna(med).to_numpy()
print(len(feats), "features |", len(tr), len(va), len(te), "linhas (treino/val/teste)")
dfe[[c for c in feats if c.startswith(("cartao_", "valor_", "hora_", "categoria_"))]].describe().T.head(12)

In [ ]:
# 3. Modelos (Pipelines: pré-processamento + balanceamento só no treino)
zoo = models.ModelZoo(random_state=42, n_jobs=-1)
pos_weight = (y_tr == 0).sum() / (y_tr == 1).sum()
pipes = models.get_supervised_pipelines(zoo, pos_weight, "class_weight",
                                        only=["Regressao_Logistica", "HistGradientBoosting", "Random_Forest"])
res = {}
for name, p in pipes.items():
    p.fit(X_tr, y_tr)
    res[name] = ev.evaluate_scores(y_va, p.predict_proba(X_va)[:, 1])
pd.DataFrame(res).T[["pr_auc", "roc_auc", "precisao_top1pct", "recall_top1pct"]].sort_values("pr_auc", ascending=False)

In [ ]:
# 4. Modelo escolhido NA VALIDAÇÃO + limiar por CUSTO (validação) + resultado no TESTE
best = max(res, key=lambda n: res[n]["pr_auc"])
s_va, s_te = pipes[best].predict_proba(X_va)[:, 1], pipes[best].predict_proba(X_te)[:, 1]
REVIEW_COST = 5.0   # custo de analisar 1 alerta
thr = ev.pick_threshold("cost", y_va, s_va, amt_va, review_cost=REVIEW_COST)
m = ev.evaluate_scores(y_te, s_te, thr, amount=amt_te, review_cost=REVIEW_COST)
print(best, "| limiar", round(thr, 4))
print({k: round(m[k], 3) for k in ("pr_auc", "precisao", "recall", "f1_score", "mcc", "alertas", "economia_pct")})
print("PR-AUC teste: %.3f [%.3f, %.3f]" % ev.bootstrap_ci(y_te, s_te, n_boot=200))

In [ ]:
# 5. Drift treino -> teste (PSI)
drift.drift_report(X_tr, X_te).head(8)

In [ ]:
# 6. Pipeline completo (todos os modelos, ensemble, relatórios, modelo empacotado)
import main
from src.config import load_config
out = main.run(load_config(None, n_samples=30_000, output_dir="outputs_notebook", n_bootstrap=100))
out["comparison"][["pr_auc_val", "pr_auc", "precisao", "recall", "economia_pct"]].round(3)

In [ ]:
# 7. Pontuar transações novas com o modelo empacotado
det = out["detector"]
novas, historico = df.tail(300), df.iloc[:-300]
det.score(novas.drop(columns="Class"), history=historico).sort_values("prob_fraude", ascending=False).head(10)[
    [c for c in ("Time", "id_cartao", "Amount", "prob_fraude", "faixa_risco") if c in df.columns or c in ("prob_fraude", "faixa_risco")]]